In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *
from rapidfuzz import process
from rapidfuzz import fuzz

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


In [4]:
file_path = os.path.join(output_folder, "Fern_matched_v2_raw_db_station.csv")
match_2_pd = pd.read_csv(file_path)


file_path = os.path.join(output_folder, "station_location_compare.csv")
station_match = pd.read_csv(file_path)

not_match_station = station_match[pd.isna(station_match['db_matched_name'])]
not_match_station_name =not_match_station['station_name']
not_match_station_inset_batch2 = match_2_pd[match_2_pd["station_name"].isin(not_match_station_name)]

filtered = not_match_station_inset_batch2[not_match_station_inset_batch2["db_var_match"].notna()]

# Update 'Has_inserted' directly in the original DataFrame using the subset's index
match_2_pd.loc[filtered.index, 'Has_inserted'] = 'Yes'

## Update this into the csv.files
# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_matched_v2_raw_db_station.csv")
match_2_pd.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")


✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_matched_v2_raw_db_station.csv


In [5]:
subset_no = match_2_pd[match_2_pd["Has_inserted"] == "No"].copy()
subset_no

# === Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_manually_matched_v3_raw_db_station.csv")
subset_no.to_csv(output_file, index=False)
print(f"✅ Matching complete! Saved to: {output_file}")


✅ Matching complete! Saved to: /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_manually_matched_v3_raw_db_station.csv


In [6]:
subset_batch4 = subset_no

print(subset_batch4['variable'].drop_duplicates())

engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


query = sa.text("""
    SELECT net_var_name, standard_name, short_name unit
    FROM meta_vars
    WHERE network_id = 11;
""")

with engine.connect() as conn:
    var_db = pd.read_sql(query, conn)
    
var_db

1              Air Temp
8         Gd Temp 75 cm
9         Gd Temp 50 cm
10        Gd Temp 25 cm
11             Sfc Temp
12           Air Temp 2
13        Water Content
14               WC_cal
17       Snow depth raw
20            Soil Temp
28               Temp 2
29                 RH 2
30              DewPt 2
31     Snow sensor Temp
66                Rain2
125            Rain raw
145          Snow depth
Name: variable, dtype: object


,net_var_name,standard_name,unit
0,RHx,relative_humidity,relative_humidity_maximum
1,WindSpeedms,wind_speed,wind_speed_mean
2,Wind_n,wind_speed,wind_speed_minimum
3,RH,relative_humidity,relative_humidity_mean
4,DewPtC,dew_point_temperature,dew_point_temperature_mean
5,Pressurembar,air_pressure,air_pressure_point
6,Rainmm,lwe_thickness_of_precipitation_amount,lwe_thickness_of_precipitation_amount_sum
7,Tm,air_temperature,air_temperature_mean
8,Tx,air_temperature,air_temperature_maximum
9,Tn,air_temperature,air_temperature_minimum


In [7]:
import re

# Define mapping for depth -> db_var_match
depth_map = {
    "5 cm": "Water_Content_m3_m3_5_cm",
    "15 cm": "Water_Content_m3_m3_15_cm",
    "30 cm": "Water_Content_m3_m3_30_cm",
}

for depth, db_var in depth_map.items():
    mask = (
        subset_batch4["variable"].eq("Water Content")
        & subset_batch4["unit_raw"].str.match(fr"m.?/m.? {depth}", na=False)
    )
    
    subset_batch4.loc[mask, "db_var_match"] = db_var
    
    # get unit from var_db if exists
    match = var_db.loc[var_db["net_var_name"] == db_var, "unit"]
    if not match.empty:
        subset_batch4.loc[mask, "unit_db"] = match.iloc[0]

# === 9. Save the updated CSV ===
output_file = os.path.join(output_folder, "Fern_manually_matched_v3_raw_db_station.csv")
subset_batch4.to_csv(output_file, index=False)


### Generate rows and insert the water content

In [72]:
filter_batch4 = subset_batch4[subset_batch4["db_var_match"].notna()].reset_index(drop=True)
filter_batch4

from row_generate_func import *

data_path = '/fern_data/FERNNorth2024_VF/WxData24'

all_rows = []

for i, row in filter_batch4.iterrows():
    station = row["station_name"]
    variable = row["variable"]

    print(f"\n🔍 [{i+1}/{len(filter_batch4)}] Processing {station} — {variable}")

    new_rows = extract_new_data_rows(row, data_path)
    all_rows.extend(new_rows)

    if len(new_rows) > 0:
        print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
    else:
        print("  ⚠️ No new records found (matches DB time range exactly).")


🔍 [1/40] Processing BarrenWx — Water Content
  ✅ 28367 new records found outside DB time range.

🔍 [2/40] Processing BarrenWx — Water Content
  ✅ 24376 new records found outside DB time range.

🔍 [3/40] Processing BarrenWx — Water Content
  ✅ 28367 new records found outside DB time range.

🔍 [4/40] Processing BlackhawkWx — Water Content
  ✅ 62899 new records found outside DB time range.

🔍 [5/40] Processing BlackhawkWx — Soil Temp
  ✅ 4153 new records found outside DB time range.

🔍 [6/40] Processing BoulderWx — Water Content
  ✅ 62214 new records found outside DB time range.

🔍 [7/40] Processing BoulderWx — Soil Temp
  ✅ 679 new records found outside DB time range.

🔍 [8/40] Processing BowronPit — Water Content
  ✅ 55387 new records found outside DB time range.

🔍 [9/40] Processing BowronPit — Soil Temp
  ✅ 1990 new records found outside DB time range.

🔍 [10/40] Processing ChapmanWx — Water Content
  ✅ 28393 new records found outside DB time range.

🔍 [11/40] Processing ChapmanWx — 

In [76]:
# from crmprtd.infer import infer

# rows_insert = all_rows

# for i in range(0, len(rows_insert), 10000):  # test in chunks of 100
#     chunk = rows_insert[i:i+100]
#     try:
#         infer(session, chunk)
#         print(f"✅ Rows {i}–{i+len(chunk)-1} OK")
#     except ValueError as e:
#         print(f"❌ Rows {i}–{i+len(chunk)-1} failed: {e}")
#         break  # stop and debug this smaller chunk

### Auto insert failed
from auto_insert_func import *

rows_insert = all_rows
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = '/workspaces/crmprtd/fern/all_stations_insert/fern_all_station_log.log'
insert_rows(rows_insert, log_file_path, db_url)


{"asctime": "2025-11-13 00:20:19,904", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = IBB2"}
{"asctime": "2025-11-13 00:20:19,979", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-13 00:20:20,004", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'IBB2', 54.81127, -126.92067, ('diagnostic', False)) -> <pycds.orm.tables.History object at 0xffff57d50410>"}
{"asctime": "2025-11-13 00:20:20,081", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: get_network ('FLNRO-FERN',) -> <pycds.orm.tables.Network object at 0xffff39ae4ad0>"}
{"asctime": "2025-11-13 00:20:20,081", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = Barren"}
{"asctime": "2025-11-13 00:20:20,086", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching

{'successes': 927415,
 'skips': 319961,
 'failures': 0,
 'insertions_per_sec': 1457.25}